Create a machine learning model to predict if a bank client is about to cancel the account.

In [13]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv('./data/Churn.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           9091 non-null   float64
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(3), int64(8), str(3)
memory usage: 1.1 MB


In [3]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2.0,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1.0,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8.0,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1.0,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2.0,125510.82,1,1,1,79084.10,0


The column 'Exited' shows if the costumer cancelled (1) or not (0). It is the model prediction objective.

## 1 - Cleaning

In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
df['CustomerId'].duplicated().sum()

np.int64(0)

There is no duplicated row or 'CustomerId'.

The columns 'RowNumber', 'CustomerId' and 'Surname' are not relevant for model training and can be discarded.

In [6]:
df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

## 2 - Preparing

### 2.1 - Class Imbalance Analysis

In [7]:
df['Exited'].value_counts(normalize=True)

Exited
0    0.7963
1    0.2037
Name: proportion, dtype: float64

Only 20% of the values are 1. This will not be balanced now to compare the model before and after considering the imbalance.

### 2.2 - Encode Cathegorical Characteristics

In [8]:
df['Geography'].unique()

<StringArray>
['France', 'Spain', 'Germany']
Length: 3, dtype: str

In [9]:
df['Gender'].unique()

<StringArray>
['Female', 'Male']
Length: 2, dtype: str

In [10]:
df = pd.get_dummies(df, drop_first=True)
df.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain,Gender_Male
0,619,42,2.0,0.00,1,1,1,101348.88,1,False,False,False
1,608,41,1.0,83807.86,1,0,1,112542.58,0,False,True,False
2,502,42,8.0,159660.80,3,1,0,113931.57,1,False,False,False
3,699,39,1.0,0.00,2,0,0,93826.63,0,False,False,False
4,850,43,2.0,125510.82,1,1,1,79084.10,0,False,True,False


## 3 - Model - Imbalance
Model without correcting the target imbalance.

### 3.1 - Splitting

In [14]:
# split df into train (0.6), validation (0.2) and test (0.2)
df_train, df_val_test = train_test_split(df, test_size=0.4, random_state=123)
df_val, df_test = train_test_split(df_val_test, test_size=0.5, random_state=123)

features_train = df_train.drop('Exited', axis=1)
target_train = df_train['Exited']

features_val = df_val.drop('Exited', axis=1)
target_val = df_val['Exited']

features_test = df_test.drop('Exited', axis=1)
target_test = df_test['Exited']

### 3.2 - RandomForestClassifier